In [22]:
import pandas as pd
import numpy as np
import importlib
import gc
import sys
import warnings
sys.path.append('./scripts')  
import preprocesamiento
import feature_engineering
import model_lgb
importlib.reload(preprocesamiento)
importlib.reload(model_lgb)
importlib.reload(feature_engineering)
warnings.filterwarnings("ignore")

# Experimento 7: 
- LGBM
- Estandarizacion del target
- Usando funcion entrenamiento: quito semillerio para ir mas rapido sino no termino mas.
- Agrego variables: agrego los ceros que dijo el profesor.
- Pesos: logaritmo
- sqlite:///optuna_studies_v21.db
- Kaggle =  


##### Levantamos el dataset con target ya calculado

In [2]:
df = pd.read_csv("./datasets/periodo_x_producto_con_target_transformado_201912.csv", sep=',', encoding='utf-8')
print("Dataset sin transformar tenia esto: (31362, 19)")
df.shape

Dataset sin transformar tenia esto: (31362, 19)


(31362, 35)

In [3]:
columnas_baseline = df.columns.tolist()
columnas_baseline

['product_id',
 'periodo',
 'nacimiento_producto',
 'muerte_producto',
 'mes_n',
 'total_meses',
 'producto_nuevo',
 'ciclo_de_vida_inicial',
 'cat1',
 'cat2',
 'cat3',
 'brand',
 'sku_size',
 'stock_final',
 'tn',
 'plan_precios_cuidados',
 'cust_request_qty',
 'cust_request_tn',
 'target',
 'tn_mean',
 'tn_std',
 'tn_zscore',
 'stock_final_mean',
 'stock_final_std',
 'stock_final_zscore',
 'cust_request_qty_mean',
 'cust_request_qty_std',
 'cust_request_qty_zscore',
 'cust_request_tn_mean',
 'cust_request_tn_std',
 'cust_request_tn_zscore',
 'tn_log',
 'stock_final_log',
 'cust_request_qty_log',
 'cust_request_tn_log']

##### Preprocesamiento a la minima expresión :)

In [4]:
df = feature_engineering.create_category_features_cat1(df)
df = feature_engineering.create_category_features_cat2(df)
df = feature_engineering.create_category_features_cat3(df)

In [5]:
# ##### aplicamos OHE
df = preprocesamiento.aplicarOHE(df)
df.shape

(31362, 187)

### Feature Engineering

##### Neural Prophet

In [6]:
neural_prophet_fe = pd.read_csv("./datasets/features_neuralprophet_completo.csv", sep=',', encoding='utf-8')
neural_prophet_fe['ds'] = pd.to_datetime(neural_prophet_fe['ds'], errors='coerce')
# Versión alternativa más robusta:
neural_prophet_fe['periodo'] = neural_prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
neural_prophet_fe = neural_prophet_fe[['periodo', 'product_id', 'trend', "season_yearly", "season_monthly"]]
df = df.merge(neural_prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 190)

##### Prophet

In [7]:
prophet_fe = pd.read_csv("./datasets/prophet_features_tn_zscore.csv", sep=',', encoding='utf-8')
prophet_fe['ds'] = pd.to_datetime(prophet_fe['ds'], errors='coerce')
prophet_fe['periodo'] = prophet_fe['ds'].apply(
    lambda x: x.year * 100 + x.month if pd.notnull(x) else None
)
prophet_fe = prophet_fe[['periodo', 'product_id', 'trend_add', "yearly_add", "additive_terms", 'trend_mult', 'yearly_mult', 'multiplicative_terms']]
df = df.merge(prophet_fe, on=['periodo', 'product_id'], how='left')
df.shape

(31362, 196)

##### FE Moviles

In [8]:
df = feature_engineering.get_lags(df, "tn", 201912)
df = feature_engineering.get_delta_lags(df, "tn", 24)
df = feature_engineering.get_rolling_means(df, "tn", 201912)
df = feature_engineering.get_rolling_stds(df, "tn", 201912)
df = feature_engineering.get_rolling_mins(df, "tn", 201912)
df = feature_engineering.get_rolling_maxs(df, "tn", 201912)
df = feature_engineering.get_rolling_medians(df, "tn", 201912)
df = feature_engineering.get_rolling_skewness(df, "tn", 201912)
df = feature_engineering.get_autocorrelaciones(df, "tn", 201912)
df.shape

(31362, 758)

In [9]:
df = feature_engineering.get_lags(df, "cust_request_qty", 201912)
df = feature_engineering.get_delta_lags(df, "cust_request_qty", 24)
df = feature_engineering.get_rolling_means(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_stds(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_mins(df, "cust_request_qty", 201912)
df = feature_engineering.get_rolling_maxs(df, "cust_request_qty", 201912)
df.shape

(31362, 1213)

In [10]:
df = feature_engineering.get_lags(df, "stock_final", 201912)
df = feature_engineering.get_delta_lags(df, "stock_final", 24)
df = feature_engineering.get_rolling_means(df, "stock_final", 201912)
df = feature_engineering.get_rolling_stds(df, "stock_final", 201912)
df = feature_engineering.get_rolling_mins(df, "stock_final", 201912)
df = feature_engineering.get_rolling_maxs(df, "stock_final", 201912)
df.shape

(31362, 1668)

Features Diana

In [11]:
df = feature_engineering.calcular_diferencia_con_medias_moviles(df)
df = feature_engineering.calcular_ratios_con_medias_moviles(df)
df.shape

(31362, 1704)

##### FE Moviles sobre otras variables

In [56]:
# #  stock final
# df = feature_engineering.get_lagsEspecificos(df, col='stock_final_zscore')
# df = feature_engineering.get_delta_lags_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_means_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_stds_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_medians_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_mins_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='stock_final_zscore')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='stock_final_zscore')

#  cust_request_qty
# df = feature_engineering.get_lagsEspecificos(df, col='cust_request_qty')
# df = feature_engineering.get_delta_lags_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_means_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_stds_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_mins_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_maxs_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_medians_especificos(df, col='cust_request_qty')
# df = feature_engineering.get_rolling_skewness_especificos(df, col='cust_request_qty')

##### FE Calendario

In [12]:
df = feature_engineering.generar_ids(df)
df = feature_engineering.get_componentesTemporales(df)
df = feature_engineering.get_anomaliasPoliticas(df)
# df = feature_engineering.descomposicion_serie_temporal(df, col='tn')
df.shape

(31362, 1729)

##### FE sobre FE

In [13]:
df = feature_engineering.chatGPT_features_serie(df, "tn")
df = feature_engineering.mes_con_feriado(df)
df.shape

(31362, 1758)

##### Variables Exogenas

In [14]:
df = feature_engineering.get_dolar(df)
df = feature_engineering.get_IPC(df)
df['ipc'] = df['ipc'].str.replace(',', '.').astype(float)
df['dolar'] = df['dolar'].str.replace(',', '.').astype(float)
# df.drop(columns=['ds'], inplace=True)
df.fillna(0, inplace=True) ##### EXPERIMENTAR
df = feature_engineering.correlacion_exogenas(df)
df = feature_engineering.get_mes_receso_escolar(df)
df.shape

(31362, 1761)

##### Nuevas FE

In [15]:
df = feature_engineering.create_ratio_features(df)
df = feature_engineering.enhance_lifecycle_features(df)
# df = feature_engineering.create_category_features(df)
df = feature_engineering.create_regime_features(df)
df = feature_engineering.create_nonlinear_trends(df)
df = feature_engineering.create_temporal_interactions(df)
df = feature_engineering.create_asymmetric_window_features(df)
df = feature_engineering.recomendaciones_deepseek(df)
df = feature_engineering.get_nuevas_features(df)
df.shape

(31362, 1798)

##### Ceros

In [16]:
df = feature_engineering.agregar_ceros_consecutivos_atras(df)
df = feature_engineering.agregar_no_ceros_consecutivos_atras(df)
df = feature_engineering.agregar_ceros_ultimos_n_meses(df, ventanas=[1,2,3,4,5,6,12])
df = feature_engineering.agregar_min_max_ult_n(df, n_list=(1,2,3,4,5,6,12))
df.shape

(31362, 1821)

##### Elimino aquellas que no sirven

In [ ]:
import json
import pandas as pd
import csv

with open("./feature_importance/v19.json") as f:
    data = json.load(f)

# Crear una lista de tuplas (feature, value)
features_values = [(feature, value) for feature, value in data.items()]

# Guardar en un archivo CSV
with open('./feature_importance/v19.csv', 'w', newline='') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(['feature', 'importance'])  # Escribir el encabezado
    writer.writerows(features_values)      # Escribir los datos

print("Archivo CSV generado exitosamente: features_values.csv")

Archivo CSV generado exitosamente: features_values.csv


In [17]:
importantes = pd.read_csv("./feature_importance/v19.csv", sep=',', encoding='utf-8')
no_importantes = importantes[importantes['importance'] == 0]
no_importantes = no_importantes[~no_importantes['feature'].isin(columnas_baseline)]
no_importantes

,feature,importance
1103,tn_rolling_std_20,0.0
1104,tn_rolling_std_22,0.0
1105,tn_rolling_std_25,0.0
1106,tn_rolling_std_26,0.0
1107,tn_rolling_std_27,0.0
...,...,...
1781,cat3_Acond Bebe,0.0
1782,tn_rolling_median_25,0.0
1783,tn_rolling_median_24,0.0
1786,tn_rolling_std_1,0.0


In [18]:
cols_a_eliminar = no_importantes.feature.unique()
print(f"Antes de eliminar: {df.shape[1]} columnas")
df = df.drop(columns=cols_a_eliminar, errors='ignore')
print(f"Después de eliminar: {df.shape[1]} columnas")

Antes de eliminar: 1821 columnas
Después de eliminar: 1139 columnas


Eliminar object/categorical columnas

In [19]:
df = df.select_dtypes(exclude=['datetime', 'datetime64', 'object'])

Train Test Split

In [20]:
train = df[df['periodo'] <= 201912]
test = df[df['periodo'] == 201912]

Entrenamiento

In [21]:
model_lgb.optimizar_con_optuna_sin_semillerio_db(train, version="v21", n_trials=500)


Para visualizar los resultados en tiempo real:
1. Abre otra terminal y ejecuta:
   optuna-dashboard sqlite:///optuna_studies_v21.db
2. Abre en tu navegador: http://127.0.0.1:8080/


[I 2025-07-11 12:32:56,635] A new study created in RDB with name: lightgbm_optimization_v21
[I 2025-07-11 12:34:03,389] Trial 0 finished with value: 1.1289816591470523 and parameters: {'num_leaves': 47, 'learning_rate': 0.2536999076681772, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8795975452591109, 'bagging_freq': 2, 'lambda_l1': 2.5348407664333426e-07, 'lambda_l2': 3.3323645788192616e-08, 'min_child_samples': 45, 'max_depth': 7, 'max_bin': 383, 'min_data_in_leaf': 21, 'extra_trees': True, 'early_stopping_rounds': 18, 'path_smooth': 0.18182496720710062, 'min_gain_to_split': 0.09170225492671691}. Best is trial 0 with value: 1.1289816591470523.


Mejor trial hasta ahora: RMSE=1.128982, Parámetros={'num_leaves': 47, 'learning_rate': 0.2536999076681772, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8795975452591109, 'bagging_freq': 2, 'lambda_l1': 2.5348407664333426e-07, 'lambda_l2': 3.3323645788192616e-08, 'min_child_samples': 45, 'max_depth': 7, 'max_bin': 383, 'min_data_in_leaf': 21, 'extra_trees': True, 'early_stopping_rounds': 18, 'path_smooth': 0.18182496720710062, 'min_gain_to_split': 0.09170225492671691}


[I 2025-07-11 12:35:10,499] Trial 1 finished with value: 1.5755570118905065 and parameters: {'num_leaves': 41, 'learning_rate': 0.05958389350068958, 'feature_fraction': 0.7727780074568463, 'bagging_fraction': 0.7873687420594125, 'bagging_freq': 7, 'lambda_l1': 1.8007140198129195e-07, 'lambda_l2': 4.258943089524393e-06, 'min_child_samples': 25, 'max_depth': 6, 'max_bin': 414, 'min_data_in_leaf': 36, 'extra_trees': False, 'early_stopping_rounds': 11, 'path_smooth': 0.6075448519014384, 'min_gain_to_split': 0.08526206184364576}. Best is trial 0 with value: 1.1289816591470523.


Mejor trial hasta ahora: RMSE=1.128982, Parámetros={'num_leaves': 47, 'learning_rate': 0.2536999076681772, 'feature_fraction': 0.892797576724562, 'bagging_fraction': 0.8795975452591109, 'bagging_freq': 2, 'lambda_l1': 2.5348407664333426e-07, 'lambda_l2': 3.3323645788192616e-08, 'min_child_samples': 45, 'max_depth': 7, 'max_bin': 383, 'min_data_in_leaf': 21, 'extra_trees': True, 'early_stopping_rounds': 18, 'path_smooth': 0.18182496720710062, 'min_gain_to_split': 0.09170225492671691}


...

[I 2025-07-12 01:19:17,094] Trial 498 finished with value: 1.7046743400883329 and parameters: {'num_leaves': 89, 'learning_rate': 0.23093919378991035, 'feature_fraction': 0.9405772409166904, 'bagging_fraction': 0.7406243321821486, 'bagging_freq': 8, 'lambda_l1': 0.05848211890476821, 'lambda_l2': 2.0423239885275915e-06, 'min_child_samples': 32, 'max_depth': 7, 'max_bin': 444, 'min_data_in_leaf': 90, 'extra_trees': False, 'early_stopping_rounds': 27, 'path_smooth': 0.5769721944603494, 'min_gain_to_split': 0.4442101024577681}. Best is trial 191 with value: 0.4882488232313609.


Mejor trial hasta ahora: RMSE=0.488249, Parámetros={'num_leaves': 93, 'learning_rate': 0.22634931577422102, 'feature_fraction': 0.9429427285714367, 'bagging_fraction': 0.7185873268490852, 'bagging_freq': 1, 'lambda_l1': 0.03529965164069097, 'lambda_l2': 1.7241089660595204e-06, 'min_child_samples': 28, 'max_depth': 7, 'max_bin': 433, 'min_data_in_leaf': 29, 'extra_trees': False, 'early_stopping_rounds': 26, 'path_smooth': 0.5629907432202811, 'min_gain_to_split': 0.45181315423785}


[I 2025-07-12 01:20:23,497] Trial 499 finished with value: 1.3229839660368021 and parameters: {'num_leaves': 94, 'learning_rate': 0.2822009988003494, 'feature_fraction': 0.9581726523808091, 'bagging_fraction': 0.7001683139361526, 'bagging_freq': 1, 'lambda_l1': 0.16740304238940457, 'lambda_l2': 0.0011456191685838778, 'min_child_samples': 26, 'max_depth': 7, 'max_bin': 491, 'min_data_in_leaf': 32, 'extra_trees': False, 'early_stopping_rounds': 25, 'path_smooth': 0.5518284914797872, 'min_gain_to_split': 0.49986344145552913}. Best is trial 191 with value: 0.4882488232313609.


Mejor trial hasta ahora: RMSE=0.488249, Parámetros={'num_leaves': 93, 'learning_rate': 0.22634931577422102, 'feature_fraction': 0.9429427285714367, 'bagging_fraction': 0.7185873268490852, 'bagging_freq': 1, 'lambda_l1': 0.03529965164069097, 'lambda_l2': 1.7241089660595204e-06, 'min_child_samples': 28, 'max_depth': 7, 'max_bin': 433, 'min_data_in_leaf': 29, 'extra_trees': False, 'early_stopping_rounds': 26, 'path_smooth': 0.5629907432202811, 'min_gain_to_split': 0.45181315423785}
Estudio guardado en: sqlite:///optuna_studies_v21.db

Mejores hiperparámetros encontrados:
num_leaves: 93
learning_rate: 0.22634931577422102
feature_fraction: 0.9429427285714367
bagging_fraction: 0.7185873268490852
bagging_freq: 1
lambda_l1: 0.03529965164069097
lambda_l2: 1.7241089660595204e-06
min_child_samples: 28
max_depth: 7
max_bin: 433
min_data_in_leaf: 29
extra_trees: False
early_stopping_rounds: 26
path_smooth: 0.5629907432202811
min_gain_to_split: 0.45181315423785


(<optuna.study.study.Study at 0x1b021e5ecd0>,
 {'num_leaves': 93,
  'learning_rate': 0.22634931577422102,
  'feature_fraction': 0.9429427285714367,
  'bagging_fraction': 0.7185873268490852,
  'bagging_freq': 1,
  'lambda_l1': 0.03529965164069097,
  'lambda_l2': 1.7241089660595204e-06,
  'min_child_samples': 28,
  'max_depth': 7,
  'max_bin': 433,
  'min_data_in_leaf': 29,
  'extra_trees': False,
  'early_stopping_rounds': 26,
  'path_smooth': 0.5629907432202811,
  'min_gain_to_split': 0.45181315423785,
  'objective': 'regression',
  'metric': 'rmse',
  'boosting_type': 'gbdt',
  'verbosity': -1})

Prediccion

In [23]:
df_future = model_lgb.semillerio_en_prediccion_con_pesos(train, test, version="v21")

In [24]:
df_future

,periodo,product_id,target,pred
30476,201912,20001,0.0,0.983107
30477,201912,20002,0.0,1.174595
30478,201912,20003,0.0,0.429366
30479,201912,20004,0.0,0.579315
30480,201912,20005,0.0,0.679257
...,...,...,...,...
31357,201912,21265,0.0,0.802648
31358,201912,21266,0.0,0.451392
31359,201912,21267,0.0,0.029572
31360,201912,21271,0.0,0.308942


Filtramos los 180 productos

In [25]:
productos_ok = pd.read_csv("https://storage.googleapis.com/open-courses/austral2025-af91/labo3v/product_id_apredecir201912.txt", sep="\t")
df_future = df_future[df_future['periodo'] == 201912]
df_future = df_future[df_future['product_id'].isin(productos_ok['product_id'].unique())]


In [26]:
df_future

,periodo,product_id,target,pred
30476,201912,20001,0.0,0.983107
30477,201912,20002,0.0,1.174595
30478,201912,20003,0.0,0.429366
30479,201912,20004,0.0,0.579315
30480,201912,20005,0.0,0.679257
...,...,...,...,...
31355,201912,21263,0.0,0.314934
31357,201912,21265,0.0,0.802648
31358,201912,21266,0.0,0.451392
31359,201912,21267,0.0,0.029572


In [27]:
df_future_copy = df_future.copy()

In [28]:
import os
ruta_archivo = f'./datasets/tn_stats_201912.csv'
    
df_stats = pd.DataFrame()

if os.path.exists(ruta_archivo) and ruta_archivo.endswith('.csv'):
    df_stats = pd.read_csv(ruta_archivo, sep=',')

df_stats

,product_id,tn_mean,tn_std
0,20001,1398.344322,293.975388
1,20002,1009.368178,299.585187
2,20003,889.004243,287.951952
3,20004,671.615383,221.310769
4,20005,644.200514,215.220300
...,...,...,...
1159,21271,0.024268,0.019484
1160,21273,0.057242,0.124272
1161,21274,0.067028,0.096980
1162,21276,0.045447,0.041380


In [29]:
df_future_copy = df_future_copy.merge(df_stats, on=['product_id'], how='left')
df_future_copy

,periodo,product_id,target,pred,tn_mean,tn_std
0,201912,20001,0.0,0.983107,1398.344322,293.975388
1,201912,20002,0.0,1.174595,1009.368178,299.585187
2,201912,20003,0.0,0.429366,889.004243,287.951952
3,201912,20004,0.0,0.579315,671.615383,221.310769
4,201912,20005,0.0,0.679257,644.200514,215.220300
...,...,...,...,...,...,...
775,201912,21263,0.0,0.314934,0.089233,0.148180
776,201912,21265,0.0,0.802648,0.089541,0.103219
777,201912,21266,0.0,0.451392,0.094659,0.100530
778,201912,21267,0.0,0.029572,0.092835,0.075836


In [30]:

df_future_copy['tn'] = df_future_copy['pred'] * df_future_copy['tn_std'] + df_future_copy['tn_mean']
df_future_copy

,periodo,product_id,target,pred,tn_mean,tn_std,tn
0,201912,20001,0.0,0.983107,1398.344322,293.975388,1687.353551
1,201912,20002,0.0,1.174595,1009.368178,299.585187,1361.259501
2,201912,20003,0.0,0.429366,889.004243,287.951952,1012.641029
3,201912,20004,0.0,0.579315,671.615383,221.310769,799.823966
4,201912,20005,0.0,0.679257,644.200514,215.220300,790.390495
...,...,...,...,...,...,...,...
775,201912,21263,0.0,0.314934,0.089233,0.148180,0.135900
776,201912,21265,0.0,0.802648,0.089541,0.103219,0.172390
777,201912,21266,0.0,0.451392,0.094659,0.100530,0.140038
778,201912,21267,0.0,0.029572,0.092835,0.075836,0.095078


Vemos cuantos negativos hay

In [31]:
df_future_copy[df_future_copy['tn'] < 0]

,periodo,product_id,target,pred,tn_mean,tn_std,tn


Reemplazamos los negativos por el promedio de ultimos 12 meses

In [ ]:
# promedio780 = model_lgb.promedio_12_meses_780p()
# df_future = df_future.merge(promedio780, on='product_id', how='left')
# df_future.drop(columns=['target','periodo'], inplace=True)
# df_future.loc[df_future['pred'] < 0, 'pred'] = df_future['tn']
# df_future



,product_id,pred,tn
0,20001,1397.305481,1454.732720
1,20002,1086.538942,1175.437142
2,20003,747.163659,784.976407
3,20004,565.799872,627.215328
4,20005,638.965713,668.270104
...,...,...,...
775,21263,0.029993,0.029993
776,21265,0.791975,0.089541
777,21266,0.094659,0.094659
778,21267,0.092835,0.092835


Guardamos el archivo

In [32]:
# df_future_copy.drop(columns=['tn'], inplace=True)
# df_future_copy.rename(columns={'pred': 'tn'}, inplace=True)
df_future_copy[['product_id', 'tn']].to_csv("./outputs/predicciones_exp_07_lgb_v9.csv", index=False, sep=',')

Ensemble

In [31]:
df_lgb = df_future_copy[['product_id', 'tn']].rename(columns={'tn': 'tn_lgb'})
df_rl = pd.read_csv("./outputs/predicciones_regresion_lineal_v1.csv", sep=',', encoding='utf-8').rename(columns={'tn': 'tn_rl'})
df_ensemble = df_lgb.merge(df_rl, on='product_id', how='left')
df_ensemble['tn'] = (df_ensemble['tn_lgb'] + df_ensemble['tn_rl']) / 2
df_ensemble[['product_id', 'tn']].to_csv("./outputs/predicciones_exp_07_lgb_v6_ensemble.csv", index=False, sep=',')

In [32]:
df_lgb = df_future_copy[['product_id', 'tn']].rename(columns={'tn': 'tn_lgb'})
df_rl = pd.read_csv("./outputs/predicciones_regresion_lineal_v1.csv", sep=',', encoding='utf-8').rename(columns={'tn': 'tn_rl'})
df_ag = pd.read_csv("./outputs/prediccion_autogluon_2ventanas.csv", sep=',', encoding='utf-8').rename(columns={'tn': 'tn_ag'})
df_ensemble = df_lgb.merge(df_rl, on='product_id', how='left')
df_ensemble = df_ensemble.merge(df_ag, on='product_id', how='left')
df_ensemble['tn'] = (df_ensemble['tn_lgb'] + df_ensemble['tn_rl'] + df_ensemble['tn_ag']) / 3
df_ensemble[['product_id', 'tn']].to_csv("./outputs/predicciones_exp_07_lgb_v6_ensemble_3models.csv", index=False, sep=',')